# Train the Drone Detector on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kcruz28/detection_system/blob/main/notebooks/train_colab.ipynb)

This notebook reuses this repo's existing `scripts/` and `configs/` (no training code is duplicated here) so you can fine-tune YOLO26 on a free Colab GPU instead of a local machine without one. It:

1. Clones this repo into the Colab VM
2. (Optionally) mounts Google Drive so the dataset/checkpoints survive across sessions
3. Installs dependencies from `pyproject.toml`
4. Downloads the dataset via `scripts/download_dataset.py`
5. Trains via `scripts/train.py` (reads `configs/train_config.yaml`)
6. Evaluates via `scripts/evaluate.py`
7. Copies `models/best.pt` to Drive (or offers a direct download) so you don't lose it

**Before running anything**: go to `Runtime > Change runtime type` and select **T4 GPU** (or better) as the hardware accelerator. Free-tier Colab GPUs are not guaranteed to be available at all times, and sessions disconnect after a period of inactivity or a fixed max duration -- keep that in mind for long training runs.

Use `scripts/train.py` directly on your own machine instead of this notebook if you have a usable local GPU; use this notebook when you don't.

## Setup

### GPU check

Confirm Colab actually gave you a GPU before doing anything else. If `torch.cuda.is_available()` prints `False`, go to `Runtime > Change runtime type`, set hardware accelerator to **T4 GPU**, save, and re-run this notebook from the top (this restarts the VM, so earlier variables/installs are lost).

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
        "save, then re-run this notebook from the top."
    )

### Clone the repo

Colab VMs are ephemeral: everything written to local disk (`/content/...`) is wiped when the runtime disconnects or recycles. The clone below, and anything else on local disk, needs to be redone each fresh session unless you saved it to Drive (see the next section).

In [ ]:
!git clone https://github.com/Kcruz28/detection_system.git
%cd detection_system

### (Optional) Mount Google Drive

Mounting Drive lets you persist the downloaded dataset and trained checkpoints across Colab sessions, so you don't have to re-download or re-train from scratch every time the runtime recycles. This is optional -- skip this cell (and the Drive copy steps later) if you don't mind starting fresh each session.

In [ ]:
USE_DRIVE = True  # set to False to skip Drive entirely and work only on local (ephemeral) disk

DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

    import os

    DRIVE_DIR = "/content/drive/MyDrive/detection_system"
    os.makedirs(f"{DRIVE_DIR}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
    print(f"Drive mounted. Persisting under: {DRIVE_DIR}")
else:
    print("Skipping Drive mount -- dataset/checkpoints will only live on the ephemeral VM disk.")

### Install dependencies

This project normally uses `uv` (`pyproject.toml` + `uv.lock`) with an isolated `.venv/`. Colab, however, runs one global/system Python for its kernel and doesn't really cooperate with a separate project venv (a `.venv`-based interpreter isn't the one the notebook kernel is running, so `uv run ...` would work but each cell would need to shell out through it, and Colab-specific integrations like `google.colab.drive`/`userdata` are only importable from Colab's system Python anyway).

The reliable fix is to still use `uv` as the installer (it's fast and respects `pyproject.toml`), but target it at Colab's system interpreter with `--system` instead of creating a venv: `uv pip install --system .` installs this project's dependencies directly into the Colab kernel's Python environment. That keeps a single dependency source of truth (`pyproject.toml`) without fighting Colab's global-env model.

If `uv` ever proves flaky in a given Colab image, the fallback (commented out below) is a plain `pip install` of the same packages -- skips `uv` entirely, at the cost of not reading versions from `pyproject.toml`.

In [ ]:
!pip install -q uv
!uv pip install --system .

# Fallback if uv gives you trouble in this Colab image -- uncomment instead of the two lines above:
# !pip install -q ultralytics opencv-python roboflow pyyaml scipy

## Dataset

### Set your Roboflow API key

Preferred: use Colab's **Secrets** manager so the key never appears in plaintext in the notebook. Click the key icon (🔑) in the left sidebar, add a secret named `ROBOFLOW_API_KEY` with your key from https://app.roboflow.com/settings/api, and toggle "Notebook access" on for this notebook. Then just run the cell below.

If you'd rather not use Secrets, comment out the `userdata` line and uncomment the manual line instead (less secure -- avoid sharing the notebook afterward with the key still in it).

In [ ]:
import os

from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

# Fallback (not recommended -- embeds the key in the notebook):
# os.environ["ROBOFLOW_API_KEY"] = "your_key_here"

print("ROBOFLOW_API_KEY set:", bool(os.environ.get("ROBOFLOW_API_KEY")))

### Download the dataset

Fill in the `--workspace`/`--project`/`--version` for the Roboflow Universe dataset you picked (visible in the dataset's URL / download dialog). This calls `scripts/download_dataset.py` exactly as documented in the repo README -- no dataset-download logic is duplicated here.

If Drive is mounted and a previous session already downloaded the dataset there, this skips re-downloading and instead copies it from Drive into the local `data/raw/` (much faster than re-pulling from Roboflow).

In [ ]:
WORKSPACE = "your-roboflow-workspace"   # <-- edit me
PROJECT = "drone-detection-dataset"      # <-- edit me
VERSION = 1                               # <-- edit me

import os
import shutil

local_data_dir = "data/raw"
drive_data_dir = f"{DRIVE_DIR}/data/raw" if DRIVE_DIR else None

if drive_data_dir and os.path.isdir(drive_data_dir) and os.listdir(drive_data_dir):
    print(f"Found existing dataset in Drive at {drive_data_dir}, copying to {local_data_dir} ...")
    shutil.copytree(drive_data_dir, local_data_dir, dirs_exist_ok=True)
else:
    print("No cached dataset found in Drive (or Drive not mounted) -- downloading from Roboflow ...")
    !python scripts/download_dataset.py --workspace "$WORKSPACE" --project "$PROJECT" --version "$VERSION"
    if drive_data_dir:
        print(f"Caching dataset to Drive at {drive_data_dir} for next time ...")
        shutil.copytree(local_data_dir, drive_data_dir, dirs_exist_ok=True)

**Important:** Roboflow's export includes its own `data.yaml`. As the repo README says, compare it against `configs/data.yaml` and update `configs/data.yaml`'s `path`/`train`/`val`/`test`/`names` to match your downloaded dataset (or point at the exported one directly) -- `scripts/train.py` and `scripts/evaluate.py` both read `configs/data.yaml`.

In [ ]:
# Quick peek at what Roboflow exported, to help you fill in configs/data.yaml above.
!find data/raw -maxdepth 2
!echo ---
!cat data/raw/*/data.yaml 2>/dev/null || echo "(no data.yaml found at that depth -- check the find output above)"

## Train

`scripts/train.py` reads hyperparameters from `configs/train_config.yaml` and supports CLI overrides for the common ones (`--epochs`, `--batch`, `--imgsz`, `--model`, `--device`). Either edit the YAML in place (cell below) or pass overrides on the command line -- both are shown.

In [ ]:
# Optional: tweak hyperparameters by rewriting configs/train_config.yaml directly.
import yaml

config_path = "configs/train_config.yaml"
with open(config_path) as f:
    train_config = yaml.safe_load(f)

# Example edits -- adjust as needed for a free-tier T4 (16GB VRAM):
# train_config["epochs"] = 50
# train_config["batch"] = 32
# train_config["model"] = "yolo26s.pt"

with open(config_path, "w") as f:
    yaml.safe_dump(train_config, f, sort_keys=False)

print(train_config)

In [ ]:
# Uses configs/train_config.yaml as edited above. To override from the CLI instead, e.g.:
#   !python scripts/train.py --epochs 50 --batch 32 --model yolo26s.pt
!python scripts/train.py

## Evaluate

Reports mAP50, mAP50-95, precision, recall, and inference FPS via `scripts/evaluate.py`, and saves the summary to `outputs/eval_metrics.json`.

In [ ]:
!python scripts/evaluate.py --weights models/best.pt --split val

## Save results

Colab wipes local disk when the session ends, so copy the trained checkpoint (and optionally the full training run directory, with loss curves/PR curves/etc.) somewhere that survives: Drive if mounted, or a direct browser download otherwise.

In [ ]:
import shutil

if DRIVE_DIR:
    shutil.copy2("models/best.pt", f"{DRIVE_DIR}/models/best.pt")
    shutil.copytree("outputs/train", f"{DRIVE_DIR}/outputs/train", dirs_exist_ok=True)
    print(f"Copied best.pt and outputs/train/ to {DRIVE_DIR}")
else:
    print("Drive not mounted -- use the direct-download cell below instead.")

If you skipped Drive (or just want a local copy right now), download the checkpoint straight to your machine via Colab's file browser (left sidebar folder icon -> navigate to `detection_system/models/best.pt` -> right-click -> Download), or run this cell:

In [ ]:
from google.colab import files

files.download("models/best.pt")